## HLS reconciliation batch processing

The following notebook is the batch processing of finding the missing S30 granules using the ESA copernicus catalog and NASA CMR as the source of information. The workflow include the steps as 
1. The main input to the HLS S30 reconciliation are the start and end date for batch processing and folder path. It is recommended to create a seperate folder `ESA_monthly_summaries` to store the generated .csv files during the workflow. For each date, there will be three main .csv files generated for audit purpoes, and a final log file for summary. The files generated are,
    - Master_ESA_Raw_[date].csv
    - HLS_Matched_Granules_[date].csv
    - HLS_Unmatched_Granules_[date].csv
    - Final log in HLS_Production_Stats_[start_date]_[end_date].csv
  
2. First, ESA catalog is searched via OData chunks for S30 granules meeting the datetime criteria and attributes such as the ContentStartDate, Cloud cover are retrieved. The granules are filtered for land tiles using <allowed_tiles.txt>
3. For every granule, HLS_ID is generated based on the Sentinel's Granule_Name, and it is used in the CMR query to retrieve the NASA attributes such as the Cloud_coverage, mean_solar_zenith_angle etc. 
4. We also need to populate with SZA for Sentinel's granules, and pysolar was used to generate the Solar Zenith Angle. 
5. We run the filter such as cloud cover > 96, and SZA > 76 to eliminate the tiles that are not required in the HLS processing pipeline. Once the granules are eliminated, we are left with remaining S30 tiles that are not processed in our pipeline.

In [ ]:
import polars as pl
import requests
import re
from datetime import datetime, timedelta, timezone
from urllib.parse import quote
from pathlib import Path
from process_esa_S2_v1 import compute_sza 

# ----------------
# Step 1: Set the initial parameters 
# ----------------
# Specify the start and end date for batch run
START_DATE = "2024-08-01"
END_DATE   = "2024-08-02"

# quality thresholds for cloud cover, solar zenith data and Odata chunk
MAX_CLOUD_COVER = 95.0
MAX_SZA = 75.5  # using 0.5 degree buffer since SZA is computed via pysolar
CHUNK_HOURS = 4
ODATA_URL = "https://catalogue.dataspace.copernicus.eu/odata/v1/Products"

# file path for the base directory
# It is suggested to create a new directory 'month' to have 
BASE_DIR = Path("./ESA_monthly_summaries/202603_esa_month")
BATCH_DIR = BASE_DIR / f"Batch_{START_DATE}_to_{END_DATE}"
BATCH_DIR.mkdir(parents=True, exist_ok=True)

TXT_FILE = Path("allowed_tiles.txt") # or BASE_DIR / "allowed_tiles.txt" 
STATS_TRACKER_FILE = BATCH_DIR / f"HLS_Production_Stats_{START_DATE}_to_{END_DATE}.csv"

# Load allowed land tiles once
print(f"Loading allowed tiles from {TXT_FILE}...")
with open(TXT_FILE, "r") as f:
    ALLOWED_TILES = {line.strip().upper() for line in f if line.strip()}

# ----------------
# Step 2. Function definition (defined ONCE outside loop)
# ----------------
def _parse_esa_item(item: dict) -> dict:
    """Extracts required metadata from a single ESA JSON payload item."""
    name = item.get("Name", "")
    tile_match = re.search(r"_T([A-Z0-9]{5})_", name)
    tile_id = tile_match.group(1) if tile_match else None
    
    content_date = item.get("ContentDate", {})
    cloud_cover = None
    
    for attr in item.get("Attributes", []):
        if attr.get("Name") == "cloudCover":
            cloud_cover = attr.get("Value")
            break

    return {
        "Granule_Name":       name,
        "Tile_ID":            tile_id,
        "Beginning_DateTime": content_date.get("Start"),
        "Ending_DateTime":    content_date.get("End"),
        "Publication_Date":   item.get("PublicationDate"),
        "Ingestion_Date":     item.get("PublicationDate"),
        "Cloud_Cover":        float(cloud_cover) if cloud_cover else None
    }

def append_to_tracker(df_stats: pl.DataFrame, file_path: Path):
    """Safely appends daily statistics to the master CSV """
    if file_path.exists():
        with open(file_path, "ab") as f:
            df_stats.write_csv(f, include_header=False)
    else:
        df_stats.write_csv(file_path)

def fetch_bulk_cmr_metadata(target_date: str) -> pl.DataFrame:
    """Bulk fetches NASA CMR data for the entire day."""
    cmr_url = "https://cmr.earthdata.nasa.gov/search/granules.umm_json"
    params = {
        "collection_concept_id": "C2021957295-LPCLOUD",
        "temporal": f"{target_date}T00:00:00Z,{target_date}T23:59:59Z",
        "page_size": 2000, 
        "page_num": 1
    }
    
    all_records = []
    with requests.Session() as session:
        while True:
            resp = session.get(cmr_url, params=params, timeout=30)
            if not resp.ok:
                print(f" CMR API Error: {resp.status_code}")
                break
                
            data = resp.json()
            items = data.get("items", [])
            if not items: break
                
            for item in items:
                umm = item.get("umm", {})
                hls_id = item.get("meta", {}).get("native-id")
                
                cloud = None
                sza = None
                uri = None
                
                for attr in umm.get("AdditionalAttributes", []):
                    name = attr.get("Name")
                    val_list = attr.get("Values", [])
                    val = val_list[0] if val_list else None
                    
                    if name == "CLOUD_COVERAGE": cloud = float(val) if val else None
                    elif name == "MEAN_SUN_ZENITH_ANGLE": sza = float(val) if val else None
                    elif name == "PRODUCT_URI": uri = val
                
                all_records.append({
                    "Expected_HLS_ID": hls_id,
                    "CMR_PRODUCT_URI": uri,
                    "CMR_Cloud_Cover": cloud,
                    "CMR_SZA": sza,
                    "CMR_Status": "FOUND"
                })
            params["page_num"] += 1
            
    return pl.DataFrame(all_records)

# ----------------
# Step 3. Pipeline functions for ESA granules
# ----------------
def fetch_esa_daily_granules(target_date: str) -> pl.DataFrame:
    """Fetch all MSIL1C granules for target_date via chunked OData."""
    print(f" Fetching ESA granules for {target_date} in {CHUNK_HOURS}-hour chunks...")
    start_of_day = datetime(*map(int, target_date.split("-")), tzinfo=timezone.utc)
    session  = requests.Session()
    session.headers.update({"Accept": "application/json"})
    records: list[dict] = []
    
    for hour in range(0, 24, CHUNK_HOURS):
        chunk_start = start_of_day + timedelta(hours=hour)
        chunk_end   = chunk_start  + timedelta(hours=CHUNK_HOURS)
        s = chunk_start.strftime("%Y-%m-%dT%H:%M:%S.000Z")
        e = chunk_end.strftime(  "%Y-%m-%dT%H:%M:%S.000Z")
        print(f"   -> Chunk {hour:02d}h-{hour+CHUNK_HOURS:02d}h : {s} lt {e}")
        odata_filter = f"Collection/Name eq 'SENTINEL-2' and contains(Name,'MSIL1C') and ContentDate/Start ge {s} and ContentDate/Start lt {e}"
        query = "&".join(["$filter=" + quote(odata_filter, safe=""), "$select=Name,ContentDate,PublicationDate,Attributes", "$expand=Attributes", "$top=1000"])
        next_url: str | None = f"{ODATA_URL}?{query}"
        chunk_count = 0
 
        while next_url:
            resp = session.get(next_url, timeout=60)
            if not resp.ok:
                print(f" WARNING: OData {resp.status_code} — {resp.text[:150]}")
                break
            data = resp.json()
            items = data.get("value", [])
            records.extend(_parse_esa_item(i) for i in items)
            chunk_count += len(items)
            next_url = data.get("@odata.nextLink")
        print(f"      {chunk_count:,} granules in this chunk (running total: {len(records):,})")
 
    if not records:
        print(" WARNING: No granules returned.")
        return pl.DataFrame()
 
    df = pl.DataFrame(records)
    null_tiles = df.filter(pl.col("Tile_ID").is_null()).height
    if null_tiles > 0:
        print(f" WARNING: {null_tiles:,} rows have null Tile_ID — dropped")
        df = df.filter(pl.col("Tile_ID").is_not_null())
 
    print(f" Final clean granule count for {target_date}: {df.height:,}")
    return df

def process_daily_reconciliation(target_date: str):
    """Processes the saved CSV for a specific date, applies filters, and hits CMR."""
    print(f"\n" + "="*50)
    print(f"Reconciling Date: {target_date}")
    print("="*50)
    
    # HIGHLIGHT: Paths now look in BATCH_DIR instead of a dynamic Pipeline folder
    master_csv = BATCH_DIR / f"Master_ESA_Raw_{target_date}.csv"
    matched_out = BATCH_DIR / f"HLS_Matched_Granules_{target_date}.csv"
    unmatched_out = BATCH_DIR / f"HLS_Unmatched_Granules_{target_date}.csv"
    
    if not master_csv.exists():
        print(f" Master CSV not found: {master_csv}. Skipping day.")
        return
        
    df = pl.read_csv(master_csv, ignore_errors=True)
    initial_count = df.height
    print(f"   -> Read {initial_count:,} rows")
    
    print(" Computing Expected HLS IDs and SZA...")
    df = df.with_columns(pl.col("Tile_ID").str.strip_chars().str.to_uppercase())
    df = df.with_columns(
        pl.struct(["Granule_Name", "Tile_ID"]).map_elements(
            lambda r: compute_sza(r["Granule_Name"], r["Tile_ID"]),
            return_dtype=pl.Float64,
        ).alias("Computed_SZA")
    )
    
    dt_str = pl.col("Granule_Name").str.extract(r"MSIL1C_(\d{8}T\d{6})", 1)
    date_part = dt_str.str.slice(0, 8)
    time_part = dt_str.str.slice(8, 7)
    parsed_date = date_part.str.to_date("%Y%m%d")
    year_str = parsed_date.dt.year().cast(pl.Utf8)
    doy_str = parsed_date.dt.ordinal_day().cast(pl.Utf8).str.pad_start(3, "0")

    df = df.with_columns(
        pl.concat_str([
            pl.lit("HLS.S30.T"), pl.col("Tile_ID"), pl.lit("."), 
            year_str, doy_str, time_part, pl.lit(".v2.0")
        ]).alias("Expected_HLS_ID")
    )
    
    df_land = df.filter(pl.col("Tile_ID").is_in(list(ALLOWED_TILES)))
    land_count = df_land.height
    print(f"   ✅ Filtered Allowed Land Tiles: {land_count:,}")

    if land_count == 0:
        return 

    print(" Fetching NASA CMR metadata...")
    df_cmr = fetch_bulk_cmr_metadata(target_date)
    df_final = df_land.join(df_cmr, on="Expected_HLS_ID", how="left")
    df_final = df_final.with_columns(pl.col("CMR_Status").fill_null("NOT_FOUND"))
    
    print(f" Applying Quality Filters (Cloud < {MAX_CLOUD_COVER}, SZA < {MAX_SZA})...")
    cloud_cond = pl.col("CMR_Cloud_Cover").fill_null(pl.col("Cloud_Cover")) < MAX_CLOUD_COVER
    sza_cond = pl.col("CMR_SZA").fill_null(pl.col("Computed_SZA")) < MAX_SZA
    
    cloud_filtered_count = df_final.filter(cloud_cond).height
    sza_filtered_count = df_final.filter(sza_cond).height
    
    df_quality_passed = df_final.filter(cloud_cond & sza_cond)
    both_filtered_count = df_quality_passed.height
    
    found_count = df_quality_passed.filter(pl.col("CMR_Status") == "FOUND").height
    not_found_count = df_quality_passed.filter(pl.col("CMR_Status") == "NOT_FOUND").height
    
    df_matched = df_quality_passed.filter(pl.col("CMR_Status") == "FOUND")
    df_unmatched = df_quality_passed.filter(pl.col("CMR_Status") == "NOT_FOUND")
    
    matched_count = df_matched.height
    unmatched_count = df_unmatched.height
    
    print("-" * 50)
    print(f" Passed Cloud Filter only    : {cloud_filtered_count:,}")
    print(f" Passed SZA Filter only      : {sza_filtered_count:,}")
    print(f" Passed BOTH Quality Filters : {both_filtered_count:,}")
    print("-" * 50)
    print(f" CMR_Status = 'FOUND'        : {found_count:,} (Out of {both_filtered_count:,})")
    print(f" CMR_Status = 'NOT_FOUND'    : {not_found_count:,} (Out of {both_filtered_count:,})")
    print("-" * 50)
    
    df_matched.write_csv(matched_out)
    df_unmatched.write_csv(unmatched_out)
    
    daily_stats = pl.DataFrame({
        "Date": [target_date],
        "Total_ESA_tiles": [df.height],
        "Land_tiles": [land_count],
        "Passed_Cloud_Filter": [cloud_filtered_count],
        "Passed_SZA_Filter": [sza_filtered_count],
        "Passed_Both_Filters": [both_filtered_count],
        "CMR_found": [found_count],
        "CMR_missing": [not_found_count]
    })
    append_to_tracker(daily_stats, STATS_TRACKER_FILE)

Loading allowed tiles from allowed_tiles.txt...


In [2]:
# ----------------
# Step 4: Main Execution 
# ----------------
start_dt = datetime.strptime(START_DATE, "%Y-%m-%d")
end_dt   = datetime.strptime(END_DATE, "%Y-%m-%d")
total_days = (end_dt - start_dt).days + 1

print("\n" + "="*50)
print(f"STARTING BATCH PIPELINE: {START_DATE} to {END_DATE}")
print("="*50)

for day_offset in range(total_days):
    current_date = (start_dt + timedelta(days=day_offset)).strftime("%Y-%m-%d")
    daily_master_file = BATCH_DIR / f"Master_ESA_Raw_{current_date}.csv"
    
    print("\n" + "*"*50)
    print(f"STARTING DAILY PROCESS: {current_date}")
    print("*"*50)
    
    # Step 1 Main pipeline (Fetch)
    print("1. Fetching initial ESA granules...")
    df_esa = fetch_esa_daily_granules(current_date)
    
    if df_esa.height > 0:
        # Save to BATCH_DIR
        df_esa.write_csv(daily_master_file)
        print(f" Saved Master ESA list to: {daily_master_file}")
        
        # Step 2 (Reconcile) 
        # Only runs AFTER the fetch and save are completely successful for this date
        process_daily_reconciliation(current_date)
    else:
        print(f" Skipping reconciliation: No granules found for {current_date}.")

# Prints production ONCE at the very end
print("\n" + "="*50)
print(f" PRODUCTION PIPELINE COMPLETE: {START_DATE} to {END_DATE}")
print(f" Master Tracking File saved to:\n   {STATS_TRACKER_FILE}")
print("="*50)


STARTING BATCH PIPELINE: 2024-08-01 to 2024-08-02

**************************************************
STARTING DAILY PROCESS: 2024-08-01
**************************************************
1. Fetching initial ESA granules...
 Fetching ESA granules for 2024-08-01 in 4-hour chunks...
   -> Chunk 00h-04h : 2024-08-01T00:00:00.000Z lt 2024-08-01T04:00:00.000Z
      2,425 granules in this chunk (running total: 2,425)
   -> Chunk 04h-08h : 2024-08-01T04:00:00.000Z lt 2024-08-01T08:00:00.000Z
      2,427 granules in this chunk (running total: 4,852)
   -> Chunk 08h-12h : 2024-08-01T08:00:00.000Z lt 2024-08-01T12:00:00.000Z
      2,095 granules in this chunk (running total: 6,947)
   -> Chunk 12h-16h : 2024-08-01T12:00:00.000Z lt 2024-08-01T16:00:00.000Z
      1,887 granules in this chunk (running total: 8,834)
   -> Chunk 16h-20h : 2024-08-01T16:00:00.000Z lt 2024-08-01T20:00:00.000Z
      2,060 granules in this chunk (running total: 10,894)
   -> Chunk 20h-24h : 2024-08-01T20:00:00.000Z lt 2